# SparkML ALS Recommendations Generator

This notebook generates personalized recommendations for cruise passengers using:
- **MySQL Database**: Passenger interests and preferences
- **RDF Knowledge Graph (Fuseki)**: Port features and activities
- **SparkML ALS (Alternating Least Squares)**: Collaborative filtering algorithm

## Workflow:
1. Load passenger interests from MySQL database
2. Load port features from RDF/SPARQL endpoint
3. Build user-item interaction matrix
4. Train ALS model using SparkML
5. Generate personalized recommendations
6. Export recommendations as JSON for frontend consumption

In [1]:
# Import required libraries
import os
import json
import sys
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
from datetime import datetime

# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, collect_list, when, isnan, isnull
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Database connectivity
import pymysql
from sqlalchemy import create_engine, text

# SPARQL/RDF imports
from SPARQLWrapper import SPARQLWrapper, JSON
import requests

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [2]:
# Initialize Spark Session
# Using local mode for notebooks (simpler and avoids cluster connectivity issues)
# To use cluster mode, change master to "spark://spark-master:7077" and ensure Spark cluster is running
spark = SparkSession.builder \
    .appName("ALS_Recommendations_Generator") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("✓ Spark Session initialized")
print(f"Spark Version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/15 07:34:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✓ Spark Session initialized
Spark Version: 3.5.0
Master: local[*]


In [3]:
# Configuration from environment variables
MYSQL_HOST = os.getenv('MYSQL_HOST', 'mysql')
MYSQL_PORT = int(os.getenv('MYSQL_PORT', '3306'))
MYSQL_DATABASE = os.getenv('MYSQL_DATABASE', 'cruise_recommender')
MYSQL_USER = os.getenv('MYSQL_USER', 'cruise_app')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', 'cruise_password')

FUSEKI_ENDPOINT = os.getenv('FUSEKI_ENDPOINT', 'http://fuseki:3030/cruise_kg/sparql')
FUSEKI_USERNAME = os.getenv('FUSEKI_USERNAME', 'admin')
FUSEKI_PASSWORD = os.getenv('FUSEKI_PASSWORD', 'admin')

SPRING_BOOT_API = os.getenv('SPRING_BOOT_API', 'http://host.docker.internal:8080/api/v1')

# Create SQLAlchemy engine for pandas compatibility
MYSQL_CONNECTION_STRING = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}?charset=utf8mb4"
mysql_engine = create_engine(MYSQL_CONNECTION_STRING, pool_pre_ping=True)

print(f"MySQL: {MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}")
print(f"Fuseki: {FUSEKI_ENDPOINT}")
print(f"Spring Boot API: {SPRING_BOOT_API}")

MySQL: mysql:3306/cruise_recommender
Fuseki: http://fuseki:3030/cruise_kg/sparql
Spring Boot API: http://host.docker.internal:8080/api/v1


In [4]:
# Connect to MySQL and load passenger interests
def load_passenger_interests(passenger_id: Optional[int] = None, user_id: Optional[int] = None):
    """Load passenger interests from MySQL database
    
    Args:
        passenger_id: Filter by passenger_id (from passengers table)
        user_id: Filter by user_id (from users table). Takes precedence if both are provided.
    """
    try:
        if user_id:
            # Query by user_id - use text() wrapper for SQLAlchemy compatibility
            query = text("""
                SELECT 
                    pi.id,
                    pi.passenger_id,
                    pi.interest_category,
                    pi.interest_keyword,
                    pi.confidence_score,
                    pi.is_explicit,
                    p.user_id
                FROM passenger_interests pi
                JOIN passengers p ON pi.passenger_id = p.id
                WHERE p.user_id = :user_id
            """)
            df = pd.read_sql(query, mysql_engine, params={'user_id': user_id})
            print(f"✓ Loaded {len(df)} passenger interests for user_id={user_id}")
        elif passenger_id:
            # Query by passenger_id - use text() wrapper for SQLAlchemy compatibility
            query = text("""
                SELECT 
                    pi.id,
                    pi.passenger_id,
                    pi.interest_category,
                    pi.interest_keyword,
                    pi.confidence_score,
                    pi.is_explicit,
                    p.user_id
                FROM passenger_interests pi
                JOIN passengers p ON pi.passenger_id = p.id
                WHERE pi.passenger_id = :passenger_id
            """)
            df = pd.read_sql(query, mysql_engine, params={'passenger_id': passenger_id})
            print(f"✓ Loaded {len(df)} passenger interests for passenger_id={passenger_id}")
        else:
            # Load all (limited)
            query = """
                SELECT 
                    pi.id,
                    pi.passenger_id,
                    pi.interest_category,
                    pi.interest_keyword,
                    pi.confidence_score,
                    pi.is_explicit,
                    p.user_id
                FROM passenger_interests pi
                JOIN passengers p ON pi.passenger_id = p.id
                LIMIT 1000
            """
            df = pd.read_sql(query, mysql_engine)
            print(f"✓ Loaded {len(df)} passenger interests")
        
        return df
    except Exception as e:
        print(f"✗ Error loading passenger interests: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Load all passenger interests
passenger_interests_df = load_passenger_interests()
if not passenger_interests_df.empty:
    print(f"\nSample interests:")
    print(passenger_interests_df.head())
else:
    print("⚠ No passenger interests found in database")

✓ Loaded 14 passenger interests

Sample interests:
   id  passenger_id interest_category interest_keyword  confidence_score  \
0  61             1       CRUISE_SHIP          Harmony               1.0   
1  62             1              PORT           Bimini               1.0   
2  63             1              PORT        Cabo Rojo               1.0   
3  64             1          ACTIVITY          Snorkel               1.0   
4  65             1          ACTIVITY            Kayak               1.0   

  is_explicit  user_id  
0     b'\x01'        3  
1     b'\x01'        3  
2     b'\x01'        3  
3     b'\x01'        3  
4     b'\x01'        3  


In [5]:
# Load port features from RDF/SPARQL endpoint
def load_port_features_from_sparql(port_code: Optional[str] = None):
    """Load port features from Fuseki SPARQL endpoint"""
    try:
        sparql = SPARQLWrapper(FUSEKI_ENDPOINT)
        if FUSEKI_USERNAME and FUSEKI_PASSWORD:
            sparql.setCredentials(FUSEKI_USERNAME, FUSEKI_PASSWORD)
        
        if port_code:
            query = f"""
                PREFIX ex: <http://example.org/port-property/>
                PREFIX schema: <http://schema.org/>
                PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
                
                SELECT DISTINCT ?port ?portCode ?portName ?property ?feature WHERE {{
                    ?port rdf:type schema:Marina .
                    ?port schema:name ?portName .
                    ?port ex:code ?portCode .
                    FILTER (?portCode = "{port_code}" || CONTAINS(?portCode, "{port_code}"))
                    {{
                        {{ ?port schema:touristAttraction ?feature . BIND("touristAttraction" AS ?property) }}
                        UNION
                        {{ ?port ex:iconicAttraction ?feature . BIND("iconicAttraction" AS ?property) }}
                        UNION
                        {{ ?port ex:activity ?feature . BIND("activity" AS ?property) }}
                        UNION
                        {{ ?port ex:excursion ?feature . BIND("excursion" AS ?property) }}
                        UNION
                        {{ ?port ex:generalInterest ?feature . BIND("generalInterest" AS ?property) }}
                        UNION
                        {{ ?port ex:mealVenueInfo ?feature . BIND("mealVenueInfo" AS ?property) }}
                        UNION
                        {{ ?port ex:restaurantInfo ?feature . BIND("restaurantInfo" AS ?property) }}
                        UNION
                        {{ ?port ex:localSpecialtyMain ?feature . BIND("localSpecialtyMain" AS ?property) }}
                        UNION
                        {{ ?port ex:localSpecialtyDessert ?feature . BIND("localSpecialtyDessert" AS ?property) }}
                        UNION
                        {{ ?port ex:culinaryIngredient ?feature . BIND("culinaryIngredient" AS ?property) }}
                        UNION
                        {{ ?port schema:servesCuisine ?feature . BIND("servesCuisine" AS ?property) }}
                    }}
                }}
                ORDER BY ?portName ?property ?feature
            """
        else:
            query = """
                PREFIX ex: <http://example.org/port-property/>
                PREFIX schema: <http://schema.org/>
                PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
                
                SELECT DISTINCT ?port ?portCode ?portName ?property ?feature WHERE {
                    ?port rdf:type schema:Marina .
                    ?port schema:name ?portName .
                    ?port ex:code ?portCode .
                    {
                        { ?port schema:touristAttraction ?feature . BIND("touristAttraction" AS ?property) }
                        UNION
                        { ?port ex:iconicAttraction ?feature . BIND("iconicAttraction" AS ?property) }
                        UNION
                        { ?port ex:activity ?feature . BIND("activity" AS ?property) }
                        UNION
                        { ?port ex:excursion ?feature . BIND("excursion" AS ?property) }
                        UNION
                        { ?port ex:generalInterest ?feature . BIND("generalInterest" AS ?property) }
                        UNION
                        { ?port ex:mealVenueInfo ?feature . BIND("mealVenueInfo" AS ?property) }
                        UNION
                        { ?port ex:restaurantInfo ?feature . BIND("restaurantInfo" AS ?property) }
                        UNION
                        { ?port ex:localSpecialtyMain ?feature . BIND("localSpecialtyMain" AS ?property) }
                        UNION
                        { ?port ex:localSpecialtyDessert ?feature . BIND("localSpecialtyDessert" AS ?property) }
                        UNION
                        { ?port ex:culinaryIngredient ?feature . BIND("culinaryIngredient" AS ?property) }
                        UNION
                        { ?port schema:servesCuisine ?feature . BIND("servesCuisine" AS ?property) }
                    }
                }
                ORDER BY ?portName ?property ?feature
                LIMIT 5000
            """
        
        sparql.setQuery(query)
        sparql.setReturnFormat(JSON)
        results = sparql.query().convert()
        
        bindings = results.get("results", {}).get("bindings", [])
        features_list = []
        for binding in bindings:
            feature = {
                "port": binding.get("port", {}).get("value", ""),
                "portCode": binding.get("portCode", {}).get("value", ""),
                "portName": binding.get("portName", {}).get("value", ""),
                "property": binding.get("property", {}).get("value", ""),
                "feature": binding.get("feature", {}).get("value", "")
            }
            features_list.append(feature)
        
        df = pd.DataFrame(features_list)
        print(f"✓ Loaded {len(df)} port features from SPARQL")
        return df
    except Exception as e:
        print(f"✗ Error loading port features from SPARQL: {e}")
        return pd.DataFrame()

# Load port features
port_features_df = load_port_features_from_sparql()
if not port_features_df.empty:
    print(f"\nSample port features:")
    print(port_features_df.head(10))
    print(f"\nUnique ports: {port_features_df['portCode'].nunique()}")
    print(f"Unique features: {port_features_df['feature'].nunique()}")
else:
    print("⚠ No port features found in RDF dataset")

✓ Loaded 4376 port features from SPARQL

Sample port features:
                                          port portCode portName  \
0  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
1  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
2  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
3  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
4  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
5  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
6  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
7  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
8  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   
9  http://example.org/port-property/port_DKAAR    DKAAR   Aarhus   

             property                                            feature  
0            activity  Coffee culture tour (Japanese siphon technique...  
1            activity  Strolling throu

In [6]:
# Build user-item interaction matrix
def build_interaction_matrix(passenger_interests_df, port_features_df):
    """Build user-item interaction matrix for ALS"""
    if passenger_interests_df.empty or port_features_df.empty:
        print("⚠ Cannot build matrix: missing data")
        return None
    
    interactions = []
    
    # Create user-item interactions based on interest matching
    for _, interest_row in passenger_interests_df.iterrows():
        passenger_id = int(interest_row['passenger_id'])
        interest_keyword = str(interest_row['interest_keyword']).lower()
        confidence = float(interest_row['confidence_score']) if pd.notna(interest_row['confidence_score']) else 1.0
        is_explicit = bool(interest_row['is_explicit']) if pd.notna(interest_row['is_explicit']) else False
        
        # Find matching port features
        matching_features = port_features_df[
            port_features_df['feature'].str.lower().str.contains(interest_keyword, na=False, regex=False)
        ]
        
        for _, feature_row in matching_features.iterrows():
            # Create unique item ID from port code and feature
            item_id = f"{feature_row['portCode']}_{feature_row['feature']}"
            item_id_hash = hash(item_id) % 1000000  # Convert to numeric ID for ALS
            
            # Rating based on confidence and match quality
            rating = 3.0  # Base rating
            if is_explicit:
                rating += 1.0  # Explicit interests get higher rating
            rating *= confidence  # Scale by confidence
            
            # Cap rating between 1 and 5
            rating = max(1.0, min(5.0, rating))
            
            interactions.append({
                'user_id': passenger_id,
                'item_id': item_id_hash,
                'item_name': feature_row['feature'],
                'port_code': feature_row['portCode'],
                'port_name': feature_row['portName'],
                'category': feature_row['property'],
                'rating': rating
            })
    
    if not interactions:
        print("⚠ No interactions found")
        return None
    
    interactions_df = pd.DataFrame(interactions)
    print(f"✓ Built interaction matrix with {len(interactions_df)} interactions")
    print(f"  Unique users: {interactions_df['user_id'].nunique()}")
    print(f"  Unique items: {interactions_df['item_id'].nunique()}")
    
    return interactions_df

# Build the interaction matrix
interaction_matrix_df = build_interaction_matrix(passenger_interests_df, port_features_df)
if interaction_matrix_df is not None and not interaction_matrix_df.empty:
    print(f"\nSample interactions:")
    print(interaction_matrix_df.head(10))

✓ Built interaction matrix with 417 interactions
  Unique users: 1
  Unique items: 320

Sample interactions:
   user_id  item_id                                          item_name  \
0        1   653051        The Bimini Road (Underwater rock formation)   
1        1   747168     Red Sea Snorkeling Boat Trip with Buffet Lunch   
2        1   862767          Red Sea Coral Reefs / Snorkeling & Diving   
3        1   929073          Snorkeling or Scuba Diving in the Red Sea   
4        1   656595                     Catamaran Sail, Snorkel & Swim   
5        1   673594            Belize Barrier Reef (Snorkeling/Diving)   
6        1   339886   Snorkeling/Diving at sunken ships or coral reefs   
7        1    48075    Snorkeling/Diving with Rays and Blacktip Sharks   
8        1   826215  Island hopping with snorkeling (Crocodile Isla...   
9        1   172156  Tiami Five Star Catamaran Sail, Snorkel & Turtles   

  port_code         port_name          category  rating  
0     BSBIM       

In [7]:
# Convert to Spark DataFrame and train ALS model
if interaction_matrix_df is not None and not interaction_matrix_df.empty:
    # Convert pandas DataFrame to Spark DataFrame
    spark_df = spark.createDataFrame(interaction_matrix_df)
    
    # Split into training and test sets
    (training, test) = spark_df.randomSplit([0.8, 0.2], seed=42)
    
    print(f"Training set: {training.count()} interactions")
    print(f"Test set: {test.count()} interactions")
    
    # Train ALS model
    als = ALS(
        maxIter=10,
        regParam=0.1,
        userCol="user_id",
        itemCol="item_id",
        ratingCol="rating",
        coldStartStrategy="drop",  # Drop users/items not in training set
        implicitPrefs=False
    )
    
    print("\n🔄 Training ALS model...")
    model = als.fit(training)
    print("✓ ALS model trained successfully")
    
    # Evaluate model
    predictions = model.transform(test)
    evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="rating",
        predictionCol="prediction"
    )
    rmse = evaluator.evaluate(predictions)
    print(f"✓ Model RMSE: {rmse:.4f}")
else:
    print("⚠ Cannot train model: no interaction matrix available")
    model = None

Training set: 338 interactions
Test set: 79 interactions

🔄 Training ALS model...
✓ ALS model trained successfully
✓ Model RMSE: 0.0650


In [8]:
# Generate recommendations for a specific passenger
def generate_recommendations_for_passenger(passenger_id: int, port_id: Optional[int] = None, num_recommendations: int = 10):
    """Generate ALS recommendations for a specific passenger"""
    if model is None:
        print("⚠ Model not trained. Cannot generate recommendations.")
        return []
    
    try:
        # Get passenger's current interests
        passenger_interests = passenger_interests_df[passenger_interests_df['passenger_id'] == passenger_id]
        
        if passenger_interests.empty:
            print(f"⚠ No interests found for passenger {passenger_id}")
            return []
        
        # Get all unique items this passenger hasn't rated yet
        user_items = spark_df.filter(col("user_id") == passenger_id).select("item_id").distinct()
        all_items = spark_df.select("item_id").distinct()
        items_to_predict = all_items.subtract(user_items)
        
        if items_to_predict.count() == 0:
            print(f"⚠ All items already rated by passenger {passenger_id}")
            # Use all items for recommendation
            items_to_predict = spark_df.select("item_id").distinct()
        
        # Create DataFrame with user_id and item_id for prediction
        from pyspark.sql.functions import lit
        user_item_df = items_to_predict.withColumn("user_id", lit(passenger_id))
        
        # Generate predictions
        predictions = model.transform(user_item_df)
        
        # Get top N recommendations
        top_recommendations = predictions \
            .orderBy(col("prediction").desc()) \
            .limit(num_recommendations)
        
        # Join with original data to get item names and details
        recommendations_with_details = top_recommendations.join(
            spark_df.select("item_id", "item_name", "port_code", "port_name", "category").distinct(),
            "item_id",
            "left"
        )
        
        # Convert to pandas for easier manipulation
        recommendations_pd = recommendations_with_details.toPandas()
        
        # Format recommendations
        recommendations_list = []
        for _, row in recommendations_pd.iterrows():
            rec = {
                "itemId": str(row['item_id']),
                "itemName": str(row['item_name']) if pd.notna(row['item_name']) else "Unknown",
                "category": str(row['category']) if pd.notna(row['category']) else "general",
                "predictedRating": float(row['prediction']) if pd.notna(row['prediction']) else 0.0,
                "portCode": str(row['port_code']) if pd.notna(row['port_code']) else "",
                "portName": str(row['port_name']) if pd.notna(row['port_name']) else "",
                "reason": f"Based on your interests and collaborative filtering (predicted rating: {row['prediction']:.2f})"
            }
            recommendations_list.append(rec)
        
        return recommendations_list
        
    except Exception as e:
        print(f"✗ Error generating recommendations: {e}")
        import traceback
        traceback.print_exc()
        return []

# Test: Generate recommendations for first passenger
if not passenger_interests_df.empty:
    first_passenger_id = int(passenger_interests_df.iloc[0]['passenger_id'])
    print(f"\n🔍 Generating recommendations for passenger ID: {first_passenger_id}")
    recommendations = generate_recommendations_for_passenger(first_passenger_id, num_recommendations=10)
    
    if recommendations:
        print(f"\n✓ Generated {len(recommendations)} recommendations:")
        for i, rec in enumerate(recommendations[:5], 1):
            print(f"  {i}. {rec['itemName']} ({rec['category']}) - Rating: {rec['predictedRating']:.2f}")
    else:
        print("⚠ No recommendations generated")


🔍 Generating recommendations for passenger ID: 1


⚠ All items already rated by passenger 1

✓ Generated 10 recommendations:
  1. Snorkeling or Sailing along the West Coast (activity) - Rating: 3.93
  2. Snorkeling/Diving in the protected bays (activity) - Rating: 3.93
  3. Carlisle Bay (Snorkeling and Shipwrecks) (iconicAttraction) - Rating: 3.93
  4. Dhow Cruise in the fjords for dolphin watching and snorkeling (activity) - Rating: 3.93
  5. Snorkeling in the natural pools and bays (activity) - Rating: 3.93


In [9]:
# Main function: Generate recommendations for a passenger at a specific port
def generate_recommendations_for_passenger_at_port(port_code: str, passenger_id: Optional[int] = None, user_id: Optional[int] = None, num_recommendations: int = 10):
    """
    Complete workflow: Generate ALS recommendations for a passenger at a specific port
    
    Args:
        port_code: Port code (e.g., "BSBIM", "NLAMS")
        passenger_id: Optional - Passenger ID from database
        user_id: Optional - User ID from database. Takes precedence if both passenger_id and user_id are provided.
        num_recommendations: Number of recommendations to generate
    
    Returns:
        List of recommendation dictionaries
    """
    print(f"\n{'='*60}")
    print(f"Generating Recommendations")
    print(f"{'='*60}")
    if user_id:
        print(f"User ID: {user_id}")
    if passenger_id:
        print(f"Passenger ID: {passenger_id}")
    print(f"Port Code: {port_code}")
    print(f"Number of Recommendations: {num_recommendations}")
    
    # Step 1: Load passenger interests
    print(f"\n[Step 1] Loading passenger interests...")
    passenger_interests = load_passenger_interests(passenger_id=passenger_id, user_id=user_id)
    
    if passenger_interests.empty:
        print("⚠ No interests found for this passenger/user")
        return []
    
    # Get the actual passenger_id from the loaded data (in case we queried by user_id)
    actual_passenger_id = int(passenger_interests.iloc[0]['passenger_id'])
    print(f"  Found {len(passenger_interests)} interests")
    print(f"  Using passenger_id: {actual_passenger_id}")
    interest_keywords = passenger_interests['interest_keyword'].tolist()
    print(f"  Interest keywords: {', '.join(interest_keywords[:5])}...")
    
    # Step 2: Load port features for the specific port
    print(f"\n[Step 2] Loading port features from RDF...")
    port_features = load_port_features_from_sparql(port_code)
    
    if port_features.empty:
        print(f"⚠ No features found for port {port_code}")
        return []
    
    print(f"  Found {len(port_features)} features for port {port_code}")
    
    # Step 3: Build interaction matrix for this passenger
    print(f"\n[Step 3] Building interaction matrix...")
    interaction_matrix = build_interaction_matrix(passenger_interests, port_features)
    
    if interaction_matrix is None or interaction_matrix.empty:
        print("⚠ Cannot build interaction matrix")
        return []
    
    # Step 4: Convert to Spark DataFrame
    print(f"\n[Step 4] Preparing Spark DataFrame...")
    spark_interaction_df = spark.createDataFrame(interaction_matrix)
    
    # Step 5: Train ALS model (or use existing if we have enough data)
    print(f"\n[Step 5] Training ALS model...")
    if spark_interaction_df.count() < 10:
        print("⚠ Not enough interactions for ALS. Using content-based filtering.")
        # Fallback: Return top features matching interests
        recommendations = []
        for keyword in interest_keywords[:5]:
            matching = port_features[
                port_features['feature'].str.lower().str.contains(keyword.lower(), na=False, regex=False)
            ]
            for _, row in matching.head(3).iterrows():
                recommendations.append({
                    "itemName": row['feature'],
                    "category": row['property'],
                    "predictedRating": 4.0,
                    "portCode": row['portCode'],
                    "portName": row['portName'],
                    "reason": f"Matches your interest in '{keyword}'"
                })
        return recommendations[:num_recommendations]
    
    # Train ALS model
    als = ALS(
        maxIter=10,
        regParam=0.1,
        userCol="user_id",
        itemCol="item_id",
        ratingCol="rating",
        coldStartStrategy="drop",
        implicitPrefs=False
    )
    
    model = als.fit(spark_interaction_df)
    print("  ✓ Model trained")
    
    # Step 6: Generate recommendations
    print(f"\n[Step 6] Generating recommendations...")
    
    # Get all items for this port
    port_items = spark_interaction_df.filter(
        col("port_code") == port_code
    ).select("item_id").distinct()
    
    if port_items.count() == 0:
        print("⚠ No items found for this port")
        return []
    
    # Create user-item pairs for prediction (use actual_passenger_id from loaded data)
    from pyspark.sql.functions import lit
    user_item_pairs = port_items.withColumn("user_id", lit(actual_passenger_id))
    
    # Generate predictions
    predictions = model.transform(user_item_pairs)
    
    # Get top recommendations
    top_predictions = predictions \
        .orderBy(col("prediction").desc()) \
        .limit(num_recommendations)
    
    # Join with item details
    recommendations_df = top_predictions.join(
        spark_interaction_df.select("item_id", "item_name", "port_code", "port_name", "category").distinct(),
        "item_id",
        "left"
    )
    
    # Convert to list
    recommendations_pd = recommendations_df.toPandas()
    recommendations_list = []
    
    for _, row in recommendations_pd.iterrows():
        rec = {
            "itemName": str(row['item_name']) if pd.notna(row['item_name']) else "Unknown",
            "category": str(row['category']) if pd.notna(row['category']) else "general",
            "predictedRating": float(row['prediction']) if pd.notna(row['prediction']) else 0.0,
            "portCode": str(row['port_code']) if pd.notna(row['port_code']) else port_code,
            "portName": str(row['port_name']) if pd.notna(row['port_name']) else "",
            "reason": f"SparkML ALS recommendation (rating: {row['prediction']:.2f})"
        }
        recommendations_list.append(rec)
    
    print(f"  ✓ Generated {len(recommendations_list)} recommendations")
    
    return recommendations_list

# Example usage
print("\n" + "="*60)
print("EXAMPLE: Generate recommendations for passenger at port")
print("="*60)

# Get first passenger and a port
if not passenger_interests_df.empty and not port_features_df.empty:
    example_passenger_id = int(passenger_interests_df.iloc[0]['passenger_id'])
    example_port_code = port_features_df.iloc[0]['portCode']
    
    example_recommendations = generate_recommendations_for_passenger_at_port(
        port_code=example_port_code,
        passenger_id=example_passenger_id,
        num_recommendations=10
    )
    
    if example_recommendations:
        print(f"\n{'='*60}")
        print("RECOMMENDATIONS:")
        print(f"{'='*60}")
        for i, rec in enumerate(example_recommendations, 1):
            print(f"{i}. {rec['itemName']}")
            print(f"   Category: {rec['category']}")
            print(f"   Predicted Rating: {rec['predictedRating']:.2f}")
            print(f"   Reason: {rec['reason']}")
            print()
    else:
        print("⚠ No recommendations generated")
else:
    print("⚠ Cannot generate example: missing data")


EXAMPLE: Generate recommendations for passenger at port

Generating Recommendations
Passenger ID: 1
Port Code: DKAAR
Number of Recommendations: 10

[Step 1] Loading passenger interests...
✓ Loaded 14 passenger interests for passenger_id=1
  Found 14 interests
  Using passenger_id: 1
  Interest keywords: Harmony, Bimini, Cabo Rojo, Snorkel, Kayak...

[Step 2] Loading port features from RDF...
✓ Loaded 10 port features from SPARQL
  Found 10 features for port DKAAR

[Step 3] Building interaction matrix...
✓ Built interaction matrix with 2 interactions
  Unique users: 1
  Unique items: 2

[Step 4] Preparing Spark DataFrame...

[Step 5] Training ALS model...
⚠ Not enough interactions for ALS. Using content-based filtering.
⚠ No recommendations generated


In [10]:
# Export function: Generate recommendations and save as JSON
def export_recommendations_json(passenger_id: int, port_code: str, output_file: str = None):
    """
    Generate recommendations and export as JSON file
    
    Args:
        passenger_id: Passenger ID
        port_code: Port code
        output_file: Optional output file path (default: recommendations_{passenger_id}_{port_code}.json)
    
    Returns:
        JSON string with recommendations
    """
    recommendations = generate_recommendations_for_passenger_at_port(
        port_code=port_code,
        passenger_id=passenger_id,
        num_recommendations=10
    )
    
    output = {
        "passengerId": passenger_id,
        "portCode": port_code,
        "recommendations": recommendations,
        "count": len(recommendations),
        "algorithm": "SparkML ALS (Alternating Least Squares)",
        "generatedAt": datetime.now().isoformat()
    }
    
    json_output = json.dumps(output, indent=2, ensure_ascii=False)
    
    if output_file:
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(json_output)
        print(f"✓ Recommendations saved to {output_file}")
    else:
        # Save to notebooks directory
        output_file = f"/home/jovyan/work/notebooks/recommendations_{passenger_id}_{port_code}.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(json_output)
        print(f"✓ Recommendations saved to {output_file}")
    
    return json_output

# Example: Export recommendations
if not passenger_interests_df.empty and not port_features_df.empty:
    example_passenger_id = int(passenger_interests_df.iloc[0]['passenger_id'])
    example_port_code = port_features_df.iloc[0]['portCode']
    
    json_output = export_recommendations_json(example_passenger_id, example_port_code)
    print("\nJSON Output:")
    print(json_output)


Generating Recommendations
Passenger ID: 1
Port Code: DKAAR
Number of Recommendations: 10

[Step 1] Loading passenger interests...
✓ Loaded 14 passenger interests for passenger_id=1
  Found 14 interests
  Using passenger_id: 1
  Interest keywords: Harmony, Bimini, Cabo Rojo, Snorkel, Kayak...

[Step 2] Loading port features from RDF...
✓ Loaded 10 port features from SPARQL
  Found 10 features for port DKAAR

[Step 3] Building interaction matrix...
✓ Built interaction matrix with 2 interactions
  Unique users: 1
  Unique items: 2

[Step 4] Preparing Spark DataFrame...

[Step 5] Training ALS model...
⚠ Not enough interactions for ALS. Using content-based filtering.
✓ Recommendations saved to /home/jovyan/work/notebooks/recommendations_1_DKAAR.json

JSON Output:
{
  "passengerId": 1,
  "portCode": "DKAAR",
  "recommendations": [],
  "count": 0,
  "algorithm": "SparkML ALS (Alternating Least Squares)",
  "generatedAt": "2026-01-15T07:35:04.847349"
}


## API Integration: Send Recommendations to Spring Boot Backend

The following cell sends recommendations directly to the Spring Boot API endpoint that the frontend consumes.

In [11]:
# Send recommendations to Spring Boot API
def send_recommendations_to_api(passenger_id: int, port_id: int, recommendations: List[Dict]):
    """
    Send recommendations to Spring Boot API endpoint
    
    Args:
        passenger_id: Passenger ID
        port_id: Port ID (from MySQL database)
        recommendations: List of recommendation dictionaries
    
    Returns:
        API response
    """
    try:
        # Format recommendations for API
        formatted_recommendations = []
        for rec in recommendations:
            formatted_rec = {
                "itemName": rec.get("itemName", ""),
                "category": rec.get("category", ""),
                "predictedRating": rec.get("predictedRating", 0.0),
                "reason": rec.get("reason", "")
            }
            formatted_recommendations.append(formatted_rec)
        
        # Prepare API payload
        payload = {
            "passengerId": passenger_id,
            "portId": port_id,
            "recommendations": formatted_recommendations,
            "count": len(formatted_recommendations),
            "algorithm": "SparkML ALS (Alternating Least Squares)"
        }
        
        # Send to API (if endpoint exists)
        # Note: This is a placeholder - you may need to create this endpoint
        # or use the existing /passengers/{passengerId}/als-recommendations endpoint
        api_url = f"{SPRING_BOOT_API}/passengers/{passenger_id}/als-recommendations?portId={port_id}"
        
        print(f"Sending recommendations to: {api_url}")
        print(f"Payload: {json.dumps(payload, indent=2)}")
        
        # Uncomment to actually send:
        # response = requests.post(api_url, json=payload, timeout=30)
        # print(f"API Response Status: {response.status_code}")
        # print(f"API Response: {response.text}")
        
        return payload
        
    except Exception as e:
        print(f"✗ Error sending to API: {e}")
        return None

# Example: Get port ID from database
def get_port_id_from_code(port_code: str):
    """Get port ID from MySQL database using port code"""
    try:
        query = text("SELECT id FROM ports WHERE port_code = :port_code LIMIT 1")
        df = pd.read_sql(query, mysql_engine, params={'port_code': port_code})
        
        if not df.empty:
            return int(df.iloc[0]['id'])
        return None
    except Exception as e:
        print(f"✗ Error getting port ID: {e}")
        return None

# Example: Complete workflow
if not passenger_interests_df.empty and not port_features_df.empty:
    example_passenger_id = int(passenger_interests_df.iloc[0]['passenger_id'])
    example_port_code = port_features_df.iloc[0]['portCode']
    
    # Generate recommendations
    recommendations = generate_recommendations_for_passenger_at_port(
        port_code=example_port_code,
        passenger_id=example_passenger_id,
        num_recommendations=10
    )
    
    if recommendations:
        # Get port ID
        port_id = get_port_id_from_code(example_port_code)
        
        if port_id:
            # Send to API
            api_payload = send_recommendations_to_api(example_passenger_id, port_id, recommendations)
            print(f"\n✓ Ready to send {len(recommendations)} recommendations to API")
        else:
            print(f"⚠ Port ID not found for code {example_port_code}")
    else:
        print("⚠ No recommendations to send")


Generating Recommendations
Passenger ID: 1
Port Code: DKAAR
Number of Recommendations: 10

[Step 1] Loading passenger interests...
✓ Loaded 14 passenger interests for passenger_id=1
  Found 14 interests
  Using passenger_id: 1
  Interest keywords: Harmony, Bimini, Cabo Rojo, Snorkel, Kayak...

[Step 2] Loading port features from RDF...
✓ Loaded 10 port features from SPARQL
  Found 10 features for port DKAAR

[Step 3] Building interaction matrix...
✓ Built interaction matrix with 2 interactions
  Unique users: 1
  Unique items: 2

[Step 4] Preparing Spark DataFrame...

[Step 5] Training ALS model...
⚠ Not enough interactions for ALS. Using content-based filtering.
⚠ No recommendations to send


## Usage Instructions

### To generate recommendations for a specific passenger at a port:

```python
# Replace with actual passenger ID and port code
passenger_id = 1  # Your passenger ID from database
port_code = "BSBIM"  # Your port code (e.g., "BSBIM", "NLAMS")

# Generate recommendations
recommendations = generate_recommendations_for_passenger_at_port(
    passenger_id=passenger_id,
    port_code=port_code,
    num_recommendations=10
)

# Export as JSON
json_output = export_recommendations_json(passenger_id, port_code)
print(json_output)
```

### To integrate with frontend:

1. Run this notebook with specific passenger_id and port_code
2. The notebook will generate recommendations using SparkML ALS
3. Export the JSON output
4. The frontend can consume this JSON to replace "--HERE--" placeholder

### Notes:
- Ensure MySQL database is accessible
- Ensure Fuseki SPARQL endpoint is running
- Ensure Spark cluster is running
- Passenger must have interests in `passenger_interests` table
- Port must have features in RDF knowledge graph